In [29]:
from alpha_vantage.timeseries import TimeSeries
import requests
from bs4 import BeautifulSoup
import pandas as pd
import io
import os
import time

In [33]:
with open("api_key.txt") as file:
    API_key = file.read()

API_key = API_key.strip()
API_key

'N0YVA5RVTIR2D1OL'

In [5]:
pd.options.display.max_columns = None

## Using Alpha vantage python library

In [10]:
ts1 = TimeSeries(key=API_key)

In [11]:
ts1.get_monthly('AAPL')

({'2026-04-23': {'1. open': '254.0800',
   '2. high': '275.7700',
   '3. low': '245.7000',
   '4. close': '273.4300',
   '5. volume': '665999564'},
  '2026-03-31': {'1. open': '262.4100',
   '2. high': '266.5300',
   '3. low': '245.5100',
   '4. close': '253.7900',
   '5. volume': '900035757'},
  '2026-02-27': {'1. open': '260.0300',
   '2. high': '280.9050',
   '3. low': '255.4500',
   '4. close': '264.1800',
   '5. volume': '988325816'},
  '2026-01-30': {'1. open': '272.2550',
   '2. high': '277.8400',
   '3. low': '243.4200',
   '4. close': '259.4800',
   '5. volume': '1036170325'},
  '2025-12-31': {'1. open': '278.0100',
   '2. high': '288.6200',
   '3. low': '266.9500',
   '4. close': '271.8600',
   '5. volume': '922283649'},
  '2025-11-28': {'1. open': '270.4200',
   '2. high': '280.3800',
   '3. low': '265.3200',
   '4. close': '278.8500',
   '5. volume': '876481453'},
  '2025-10-31': {'1. open': '255.0400',
   '2. high': '277.3200',
   '3. low': '244.0000',
   '4. close': '270.

## Using Alpha vantage api

In [12]:
def return_json(url):
    # replace the "demo" apikey below with your own key from https://www.alphavantage.co/support/#api-key
    url = url
    r = requests.get(url)
    data = r.json()
    return data

In [31]:
def query_all_statements(ticker):
    """
    """
    function_names= {   "INCOME_STATEMENT":"annualReports",
                        "BALANCE_SHEET":"annualReports",
                        "CASH_FLOW":"annualReports",
                        "OVERVIEW":"EMPTY",
                        "DIVIDENDS":"data",
                        "SPLITS":"data",
                        "SHARES_OUTSTANDING":"data",
                        "EARNINGS":"annualEarnings",
                        "EARNINGS_ESTIMATES":"estimates"
                        }

    for func in function_names:
        url = f'https://www.alphavantage.co/query?function={func}&symbol={ticker}&apikey={API_key}'
        return_statement = return_json(url)

        # 1. Check for API limit or Error messages
        if "Information" in return_statement:
            print(f"⚠️ API Limit hit on {func}. Skipping...")
            time.sleep(60) # Wait a full minute if limited
            continue

        if func in ["INCOME_STATEMENT","BALANCE_SHEET","CASH_FLOW"]:
            print(f"key is {func}, value is {function_names[func]}")
            df = pd.json_normalize(return_statement.get(function_names[func])).set_index("fiscalDateEnding")
        elif func == "OVERVIEW":
            print(f"key is {func}, value is {function_names[func]}")
            df = pd.json_normalize(return_statement).set_index("Symbol")
        else:
            print(f"key is {func}, value is {function_names[func]}")
            df = pd.json_normalize(return_statement.get(function_names[func]))

        path = f"stocks/{ticker}"

        if not(os.path.exists(path)):
            os.mkdir(path)
        df.to_csv(f"./{path}/{func}.csv")
        time.sleep(15)

In [32]:
query_all_statements("TSLA")

⚠️ API Limit hit on INCOME_STATEMENT. Skipping...


KeyboardInterrupt: 

In [40]:
url = 'https://www.alphavantage.co/query?function=NEWS_SENTIMENT&tickers=AAPL&apikey={API_key}'
appl_sentiment = return_json(url)

In [41]:
appl_sentiment

{'items': '50',
 'sentiment_score_definition': 'x <= -0.35: Bearish; -0.35 < x <= -0.15: Somewhat-Bearish; -0.15 < x < 0.15: Neutral; 0.15 <= x < 0.35: Somewhat_Bullish; x >= 0.35: Bullish',
 'relevance_score_definition': '0 < x <= 1, with a higher score indicating higher relevance.',
 'feed': [{'title': 'Why Is TD Cowen Doubling Down on Amazon (AMZN), Microsoft (MSFT), and Apple (AAPL) ahead of Earnings Next Week',
   'url': 'https://www.tipranks.com/news/why-is-td-cowen-doubling-down-on-amazon-amzn-microsoft-msft-and-apple-aapl-ahead-of-earnings-next-week',
   'time_published': '20260424T153952',
   'authors': ['Radhika Saraogi'],
   'summary': "TD Cowen is bullish on Amazon, Apple, and Microsoft ahead of their upcoming earnings reports, citing accelerated AI adoption, robust cloud demand, and strong product cycles as key drivers for durable growth. The firm believes the market is underestimating the AI momentum of these Big Tech giants and foresees significant revenue streams from G